In [13]:
%pip install hyperopt setuptools

Note: you may need to restart the kernel to use updated packages.


In [14]:
import sys
import importlib.metadata as importlib_metadata
sys.modules['pkg_resources'] = importlib_metadata

import keras
import numpy as np
import pandas as pd
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

In [15]:
### Load the dataset

import pandas as pd

data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [16]:
### split the data into traning, test and validation sets 
train,test= train_test_split(data, test_size=0.2, random_state=42)
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
4665,7.3,0.17,0.36,8.20,0.028,44.0,111.0,0.99272,3.14,0.41,12.4,6
1943,6.3,0.25,0.44,11.60,0.041,48.0,195.0,0.99680,3.18,0.52,9.5,5
3399,5.6,0.32,0.33,7.40,0.037,25.0,95.0,0.99268,3.25,0.49,11.1,6
843,6.9,0.19,0.35,1.70,0.036,33.0,101.0,0.99315,3.21,0.54,10.8,7
2580,7.7,0.30,0.26,18.95,0.053,36.0,174.0,0.99976,3.20,0.50,10.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [17]:
train_x= train.drop(['quality'], axis=1).values
train_y=train[['quality']].values.ravel()

### Validation dataset

test_x= test.drop(['quality'], axis=1).values
test_y= test[['quality']].values.ravel()

### Splitting this train data into train and validation.

train_x, valid_x,train_y, valid_y= train_test_split(train_x, train_y, test_size=0.02, random_state=42)

signature = infer_signature(train_x, train_y)

In [18]:
np.mean(train_x, axis=0)

array([6.86492576e+00, 2.79192498e-01, 3.32776765e-01, 6.44795520e+00,
       4.55993748e-02, 3.50952071e+01, 1.37948685e+02, 9.94067588e-01,
       3.18998437e+00, 4.89684814e-01, 1.05124694e+01])

In [24]:
### ANN model
def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    ## Define the model Architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(1)
        ]
    )

    ## Compile the model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params['lr'],
            momentum=params['momentum']
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()]
    )

    ## Train the ANN model with lr and momentum params with MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(
            train_x, train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64
        )

        ### Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        ## Log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        ## Log the model
        signature = infer_signature(train_x, model.predict(train_x))
        mlflow.tensorflow.log_model(model, "model", signature=signature)

    return {"loss": eval_rmse, "status": STATUS_OK, 'model': model}

In [25]:
def objective(params):
    results = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y
    )
    return results

In [26]:
space = {
    "lr":hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum":hp.uniform("momentum", 0.0,1.0)
}

In [27]:
mlflow.set_experiment("Wine-quality")
with mlflow.start_run():
    trials=Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
)
    
## Fetch the details of the best run
best_run = sorted(trials.results, key=lambda x:x["loss"])[0]

## log the best model parameters, loss and model
mlflow.log_params(best)
mlflow.log_metric("eval_rmse", best_run["loss"])
mlflow.tensorflow.log_model(best_run['model'], "model", signature=signature)

## Print out the best parameters and corresponding loss
print(f"Best Parameters: {best}")
print(f"Best eval rmse: {best_run['loss']}")


/Users/vr7/Desktop/MLops/Mlflow-start/ven5/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/06 18:01:47 INFO mlflow.tracking.fluent: Experiment with name 'Wine-quality' does not exist. Creating a new experiment.


Epoch 1/3                                            

 1/60 ━━━━━━━━━━━━━━━━━━━━ 21s 372ms/step - loss: 34.1123 - root_mean_squared_error: 5.8406
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 34.3593 - root_mean_squared_error: 5.8617 - val_loss: 34.3201 - val_root_mean_squared_error: 5.8583

Epoch 2/3                                            

 1/60 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 33.1309 - root_mean_squared_error: 5.7559
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 874us/step - loss: 33.3766 - root_mean_squared_error: 5.7772 - val_loss: 33.2678 - val_root_mean_squared_error: 5.7678

Epoch 3/3                                            

 1/60 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 35.2045 - root_mean_squared_error: 5.9333
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 32.4234 - root_mean_squared_error: 5.6942 - val_loss: 32.2501 - val_root_mean_squared_error: 5.6789

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 32.0314 - root_mean_squared_error: 5.6596
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10m

2026/05/06 18:01:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                     

 1/60 ━━━━━━━━━━━━━━━━━━━━ 15s 264ms/step - loss: 37.5019 - root_mean_squared_error: 6.1239
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.8527 - root_mean_squared_error: 1.3611 - val_loss: 0.5993 - val_root_mean_squared_error: 0.7741

Epoch 2/3                                                                     

 1/60 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.5935 - root_mean_squared_error: 0.7704
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 808us/step - loss: 0.5965 - root_mean_squared_error: 0.7723 - val_loss: 0.6158 - val_root_mean_squared_error: 0.7848

Epoch 3/3                                                                     

 1/60 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.8016 - root_mean_squared_error: 0.8953
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 775us/step - loss: 0.5548 - root_mean_squared_error: 0.7449 - val_loss: 0.5213 - val_root_mean_squared_error: 0.7220

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.501

2026/05/06 18:01:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

Epoch 1/3                                                                      

 1/60 ━━━━━━━━━━━━━━━━━━━━ 34s 584ms/step - loss: 30.2959 - root_mean_squared_error: 5.5042
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12.0767 - root_mean_squared_error: 3.4752 - val_loss: 4.4574 - val_root_mean_squared_error: 2.1113

Epoch 2/3                                                                      

 1/60 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.4246 - root_mean_squared_error: 1.8506
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.4672 - root_mean_squared_error: 1.5707 - val_loss: 3.1363 - val_root_mean_squared_error: 1.7710

Epoch 3/3                                                                      

 1/60 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.4384 - root_mean_squared_error: 1.1993
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 916us/step - loss: 1.8918 - root_mean_squared_error: 1.3754 - val_loss: 2.5408 - val_root_mean_squared_error: 1.5940

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 

2026/05/06 18:02:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:28<00:00,  7.08s/trial, best loss: 0.7220247387886047]

2026/05/06 18:02:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best Parameters: {'lr': np.float64(0.05430547628180882), 'momentum': np.float64(0.29739274492920453)}
Best eval rmse: 0.7220247387886047
